# 章节实践

通过 03.03.01 到 03.03.07 的学习，我们已经掌握了 SIMD 抽象硬件架构、Kernel 入口、SIMD 编程 API、基于指针的 C 语言编程、基于 Tensor 的 C++ 编程、`TPipe/TQue` 以及静态 Tensor 编程。为了巩固所学知识，现提供以下实践练习：

实现 Ascend C Vector 算子 Axpy，算子文件名为 `apxy.asc`，核函数名为 `element_wise_compound_compute_custom`，补齐 kernel 侧代码，并完成 kernel 直调测试。

相关算法：

```text
dst = dst + src * scalar
```

要求：

- 输入输出 shape 均为 `[1, 512]`，输入输出类型均为 `half`。
- `input0` 表示 `src`，`input1` 表示初始 `dst`，`scalar` 固定为 `2.0`。
- 使用 SIMD Vector 路径完成计算，可以选择前面学过的任意一种 API 形式。
- NPU 架构使用 `dav-3510`。
- 已提供 `data_utils.h`、输入数据生成脚本和结果校验脚本，开发者无需编写这些辅助文件。

请开始你的实践，体验从理解到创造的完整开发过程。

---

环境准备：


In [ ]:
!mkdir -p Sources/03.03.08

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")


---

apxy.asc 代码补齐：


In [ ]:
%%writefile Sources/03.03.08/apxy.asc

#include <cstdint>
#include "acl/acl.h"
#include "data_utils.h"
#include "kernel_operator.h"


template <uint32_t totalLength>
__global__ __vector__ void element_wise_compound_compute_custom(
    __gm__ uint8_t* srcGm, __gm__ uint8_t* dstInGm, __gm__ uint8_t* dstGm)
{
    // TODO: 补齐 kernel 侧代码。
    // 目标公式：dst = dst + src * 2.0
    // input0 为 src，input1 为初始 dst，输出写入 dstGm。
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t totalLength = 512;
    uint32_t numBlocks = 1;
    size_t inputFileSize = totalLength * sizeof(half);
    size_t outputFileSize = totalLength * sizeof(half);

    aclInit(nullptr);
    int32_t deviceId = 0;
    aclrtSetDevice(deviceId);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    uint8_t* input0Host;
    uint8_t* input0Device;
    aclrtMallocHost((void**)(&input0Host), inputFileSize);
    aclrtMalloc((void**)&input0Device, inputFileSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile("./input/input0.bin", inputFileSize, input0Host, inputFileSize);
    aclrtMemcpy(input0Device, inputFileSize, input0Host, inputFileSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t* input1Host;
    uint8_t* input1Device;
    aclrtMallocHost((void**)(&input1Host), inputFileSize);
    aclrtMalloc((void**)&input1Device, inputFileSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile("./input/input1.bin", inputFileSize, input1Host, inputFileSize);
    aclrtMemcpy(input1Device, inputFileSize, input1Host, inputFileSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t* outputHost;
    uint8_t* outputDevice;
    aclrtMallocHost((void**)(&outputHost), outputFileSize);
    aclrtMalloc((void**)&outputDevice, outputFileSize, ACL_MEM_MALLOC_HUGE_FIRST);

    element_wise_compound_compute_custom<totalLength>
        <<<numBlocks, nullptr, stream>>>(input0Device, input1Device, outputDevice);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(outputHost, outputFileSize, outputDevice, outputFileSize, ACL_MEMCPY_DEVICE_TO_HOST);
    WriteFile("./output/output.bin", outputHost, outputFileSize);

    aclrtFree(input0Device);
    aclrtFreeHost(input0Host);
    aclrtFree(input1Device);
    aclrtFreeHost(input1Host);
    aclrtFree(outputDevice);
    aclrtFreeHost(outputHost);

    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();

    return 0;
}


---

生成 CMakeLists.txt 文件（直接执行，无需修改）


In [ ]:
%%writefile Sources/03.03.08/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510")
find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    apxy.asc
)
target_include_directories(demo PRIVATE
    ${CMAKE_CURRENT_LIST_DIR}/../../answer/03.03.08_axpy
)
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)


---

执行以下命令进行编译并验证结果：


In [ ]:
!cd Sources/03.03.08 && \
mkdir -p build
!cd Sources/03.03.08/build && \
cmake .. -DCMAKE_ASC_ARCHITECTURES=dav-3510 && \
make -j
!cd Sources/03.03.08 && \
python3 ../../answer/03.03.08_axpy/scripts/gen_data.py && \
./build/demo && \
python3 ../../answer/03.03.08_axpy/scripts/verify_result.py output/output.bin output/golden.bin


预期执行效果如下：

```text
error ratio: 0.0000, tolerance: 0.0010
test pass!
```

---

执行以下代码获取答案。


In [ ]:
!cat ./answer/03.03.08_axpy/apxy.asc
